In [ ]:
import duckdb
import random
import pandas as pd

In [ ]:
PATH = '../data_2025/fulldata_07_24_04_P_일반음식점.csv'
df = pd.read_csv(PATH, encoding='cp949')
df = df.copy()

In [ ]:
# PATH = '../data/fulldata_07_24_04_P_일반음식점.csv'
# df = pd.read_csv(PATH, encoding='cp949')
# df = df.copy()

In [ ]:
def split_address(address):
    parts = address.split()
    
    if len(parts) >= 4:
        city_name = parts[0]  # 예: 전라북도
        district = parts[1]  # 예: 남원시
        town = parts[2]  # 예: 대강면
        road_name = ' '.join(parts[3:-1])  # 예: 대강월산길 (주소의 마지막 요소 전까지 모두 도로명으로 처리)
        bungee = parts[-1]  # 예: 37-16 (주소의 마지막 요소는 항상 번지로 처리)
        
        return city_name, district, town, road_name, bungee
    else:
        return None, None, None, None, None

In [ ]:
def prepro_df(df):
    random_value = random.randint(10120, 10200)
    df = df[:random_value].loc[:, ['개방서비스명', '인허가일자', '폐업일자','영업상태명', '도로명전체주소', '사업장명', '업태구분명', '남성종사자수', '여성종사자수']]
    df['도로명전체주소'] = df['도로명전체주소'].fillna('')
    df['폐업일자'] = df['폐업일자'].fillna('운영중')
    df.loc[:, '도로명전체주소'] = [address.split(',')[0] for address in df['도로명전체주소']]

    df[['시도', '시군구', '읍면동', '도로명', '번지']] = df['도로명전체주소'].apply(lambda x: pd.Series(split_address(x)))
    df.loc[:, 'num'] = 1
    df = df.rename(columns={'도로명전체주소': '소재지', '사업장명': '시설명', '업태구분명': '구분'})
    df = df.reset_index().drop(columns='index')
    return df
    # df.to_csv(path, index=False)

In [ ]:
sorted_day_df = df.sort_values(by='인허가일자',ascending=False)
sorted_day_df['인허가일자'] = pd.to_datetime(sorted_day_df['인허가일자'], format='%Y-%m-%d', errors='coerce')
sorted_day_df['인허가일자'].fillna(method='ffill', inplace=True)

df_2025 = sorted_day_df[sorted_day_df['인허가일자'].dt.year.astype(int) == 2025].reset_index().drop('index', axis=1)
df_2024 = sorted_day_df[sorted_day_df['인허가일자'].dt.year.astype(int) == 2024].reset_index().drop('index', axis=1)
df_2023 = sorted_day_df[sorted_day_df['인허가일자'].dt.year.astype(int) == 2023].reset_index().drop('index', axis=1)
df_2022 = sorted_day_df[sorted_day_df['인허가일자'].dt.year.astype(int) == 2022].reset_index().drop('index', axis=1)
df_2021 = sorted_day_df[sorted_day_df['인허가일자'].dt.year.astype(int) == 2021].reset_index().drop('index', axis=1)
df_2020 = sorted_day_df[sorted_day_df['인허가일자'].dt.year.astype(int) == 2020].reset_index().drop('index', axis=1)

In [ ]:
sorted_day_df.head()

In [ ]:
df_2025 = prepro_df(df_2025)
df_2024 = prepro_df(df_2024)
df_2023 = prepro_df(df_2023)
df_2022 = prepro_df(df_2022)
df_2021 = prepro_df(df_2021)
df_2020 = prepro_df(df_2020)

In [ ]:
print(len(df_2020))
print(len(df_2021))
print(len(df_2022))
print(len(df_2023))
print(len(df_2024))
print(len(df_2025))

In [ ]:
df_2025[:3]

In [ ]:
df_2024[:3]

In [ ]:
df_2023[:3]

In [ ]:
df_2022[:3]

In [ ]:
df_2021[:3]

In [ ]:
df_2020[:3]

In [ ]:
df_2020.to_csv('../data_2025/restaurant_2020.csv', index=False)
df_2021.to_csv('../data_2025/restaurant_2021.csv', index=False)
df_2022.to_csv('../data_2025/restaurant_2022.csv', index=False)
df_2023.to_csv('../data_2025/restaurant_2023.csv', index=False)
df_2024.to_csv('../data_2025/restaurant_2024.csv', index=False)
df_2025.to_csv('../data_2025/restaurant_2025.csv', index=False)

In [ ]:
conn = duckdb.connect('../data_2025/database.db') # database 연결

In [ ]:
# 데이터 삭제
conn.execute('DROP TABLE IF EXISTS restaurant_2025')
conn.execute('DROP TABLE IF EXISTS restaurant_2024')
conn.execute('DROP TABLE IF EXISTS restaurant_2023')
conn.execute('DROP TABLE IF EXISTS restaurant_2022')
conn.execute('DROP TABLE IF EXISTS restaurant_2021')
conn.execute('DROP TABLE IF EXISTS restaurant_2020')

In [ ]:
# 데이터 저장
conn.execute('CREATE TABLE restaurant_2025 AS SELECT * FROM df_2025')
conn.execute('CREATE TABLE restaurant_2024 AS SELECT * FROM df_2024')
conn.execute('CREATE TABLE restaurant_2023 AS SELECT * FROM df_2023')
conn.execute('CREATE TABLE restaurant_2022 AS SELECT * FROM df_2022')
conn.execute('CREATE TABLE restaurant_2021 AS SELECT * FROM df_2021')
conn.execute('CREATE TABLE restaurant_2020 AS SELECT * FROM df_2020')

In [ ]:
# database close
conn.close()